In [ ]:
from teehr import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session

#### Start the spark session

In [ ]:
spark = create_spark_session(
    aws_access_key_id="minioadmin",
    aws_secret_access_key="minioadmin123",
    update_configs={
        "spark.hadoop.fs.s3a.aws.credentials.provider":
        "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider"
    }
)

#### Initialize a TEEHR Evaluation with read/write privileges

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark)

#### Define metrics to calculate

In [ ]:
from teehr import DeterministicMetrics as dm
from teehr import Signatures as s

SIM_METRICS = [
        s.Count(),
        s.Average(),
        dm.RelativeBias(add_epsilon=True),
        dm.NashSutcliffeEfficiency(add_epsilon=True),
        dm.KlingGuptaEfficiency(add_epsilon=True),
        dm.RootMeanStandardDeviationRatio(add_epsilon=True)
]
METRIC_COL_NAMES = [metric.output_field_name for metric in SIM_METRICS]
SIM_GROUPBY = ["primary_location_id", "configuration_name", "variable_name", "unit_name"]

#### Execute the metrics calculation and join the locations data to the resulting table

In [ ]:
metrics = ev.table("sim_joined_timeseries").aggregate(
    group_by=SIM_GROUPBY,
    include_metrics=SIM_METRICS
).order_by(SIM_GROUPBY).add_geometry()

In [ ]:
metrics.to_sdf().show()

#### Write the to warehouse as a new table

In [ ]:
metrics.write(
    table_name="sim_metrics_by_location",
    write_mode="create_or_replace"
)

#### Add properties to the metrics table that are used by the dashboard

In [ ]:
table_name = "sim_metrics_by_location"
properties = {
    "description": "Simulation metrics by location ID",
    "group_by": ", ".join(SIM_GROUPBY),
    "metrics": ", ".join(METRIC_COL_NAMES)
}

In [ ]:
for key, value in properties.items():
    ev.spark.sql(f"""
    ALTER TABLE iceberg.teehr.{table_name} SET TBLPROPERTIES ('{key}' = '{value}')
    """)

In [ ]:
spark.stop()